In [1]:
from transformers import BertForQuestionAnswering
from transformers import BertTokenizer
import torch

In [6]:
model_name = "bert-large-uncased-whole-word-masking-finetuned-squad"

In [7]:
tokenizer = BertTokenizer.from_pretrained(model_name)

In [8]:
model = BertForQuestionAnswering.from_pretrained(model_name)

Some weights of the model checkpoint at bert-large-uncased-whole-word-masking-finetuned-squad were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [17]:
sunset_motors_context = "Sunset Motors is a renowned automobile dealership that has been a cornerstone of the automotive industry since its establishment in 1978. Located in the picturesque town of Crestwood, nestled in the heart of California's scenic Central Valley, Sunset Motors has built a reputation for excellence, reliability, and customer satisfaction over the past four decades. Founded by visionary entrepreneur Robert Anderson, Sunset Motors began as a humble, family-owned business with a small lot of used cars. However, under Anderson's leadership and commitment to quality, it quickly evolved into a thriving dealership offering a wide range of vehicles from various manufacturers. Today, the dealership spans over 10 acres, showcasing a vast inventory of new and pre-owned cars, trucks, SUVs, and luxury vehicles. One of Sunset Motors' standout features is its dedication to sustainability. In 2010, the dealership made a landmark decision to incorporate environmentally friendly practices, including solar panels to power the facility, energy-efficient lighting, and a comprehensive recycling program. This commitment to eco-consciousness has earned Sunset Motors recognition as an industry leader in sustainable automotive retail. Sunset Motors proudly offers a diverse range of vehicles, including popular brands like Ford, Toyota, Honda, Chevrolet, and BMW, catering to a wide spectrum of tastes and preferences. In addition to its outstanding vehicle selection, Sunset Motors offers flexible financing options, allowing customers to secure affordable loans and leases with competitive interest rates."
print(sunset_motors_context)

Sunset Motors is a renowned automobile dealership that has been a cornerstone of the automotive industry since its establishment in 1978. Located in the picturesque town of Crestwood, nestled in the heart of California's scenic Central Valley, Sunset Motors has built a reputation for excellence, reliability, and customer satisfaction over the past four decades. Founded by visionary entrepreneur Robert Anderson, Sunset Motors began as a humble, family-owned business with a small lot of used cars. However, under Anderson's leadership and commitment to quality, it quickly evolved into a thriving dealership offering a wide range of vehicles from various manufacturers. Today, the dealership spans over 10 acres, showcasing a vast inventory of new and pre-owned cars, trucks, SUVs, and luxury vehicles. One of Sunset Motors' standout features is its dedication to sustainability. In 2010, the dealership made a landmark decision to incorporate environmentally friendly practices, including solar p

In [23]:
def faq_bot(question):
    # The context containing the information about the company
    context=sunset_motors_context
    # Turning to embeddings that will be readable by the model
    input_ids=tokenizer.encode(question, context)
    # Turning the words from digits to tokens
    tokens=tokenizer.convert_ids_to_tokens(input_ids)
    # Finding the exact position of the first sep token
    sep_idx=input_ids.index(tokenizer.sep_token_id)
    # count tokens in the question
    num_seg_a = sep_idx+1
    # count tokens in the context
    num_seg_b = len(input_ids) - num_seg_a
    # creating segment ids
    segment_ids = [0]*num_seg_a + [1]*num_seg_b
    # Training the model and giving in torch
    output=model(torch.tensor([input_ids]), token_type_ids = torch.tensor([segment_ids]))
    answer_start = torch.argmax(output.start_logits)
    answer_end = torch.argmax(output.end_logits)
    if answer_end>=answer_start:
        answer = ' '.join(tokens[answer_start:answer_end+1])
    else:
        print("I do not know the answer to your question, can you ask another question?")
    corrected_answer = ''
    for word in answer.split():
        if word[0:2] == "##":
            corrected_answer += word[2:]
        else:
            corrected_answer += ' ' + word
    return corrected_answer

In [24]:
faq_bot("where is the dealership located?")

' crestwood'

In [25]:
faq_bot("what make of cars are available?")

' ford , toyota , honda , chevrolet , and bmw'

In [26]:
faq_bot("how large is the dealership?") 

' 10 acres'